In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_project.silver.actions (
    action_id BIGINT,
    name STRING,
    date_from TIMESTAMP,
    date_to TIMESTAMP,
    ingested_at TIMESTAMP
) USING DELTA

In [0]:
%python
from delta.tables import DeltaTable
from pyspark.sql import functions as F

table_name = f"databricks_project.silver.actions"
dim_table = DeltaTable.forName(spark, table_name)

df = spark.sql('''
    SELECT action_id, name, ingested_at, 
        ingested_at as date_from,
        null as date_to
    FROM databricks_project.bronze.actions
''')

merge_condition = """
    target.action_id = source.action_id 
    AND target.name != source.name
"""

(dim_table.alias("target")
    .merge(df.alias("source"), merge_condition)
    .whenMatchedUpdate(set={
        "date_to": F.col("source.date_from")
    })
    .whenNotMatchedInsertAll()
    .execute()
)